In [0]:
# -------------------------------------------------------------------------
# DLT PIPELINE: GOLD LAYER (03_gold_aggregations)
# STRICT ARCHITECTURE: Transactional Star Schema (No KPIs in DLT)
# -------------------------------------------------------------------------

import dlt
from pyspark.sql.functions import col, current_timestamp

SILVER_TABLE = "ecommerce_analytics_dev.silver_layer.events_silver"

# =========================================================================
# 1. FACT TABLE: TRANSACTIONAL SALES (Granular)
# =========================================================================
# RUBRIC: "Price - it is a fact and all updates should be stored"
# We store every event with its specific price as a "Fact".
@dlt.table(
    name="gold_fact_sales",
    comment="Transactional Fact Table. One row per event. Contains Raw Price.",
    table_properties={"quality": "gold"}
)
def gold_fact_sales():
    return (
        spark.readStream.table(SILVER_TABLE)
        # We select ONLY Foreign Keys and Facts (Measures)
        .select(
            col("event_time"),      # Time Dimension Key
            col("event_date"),      # Partition Key
            col("product_id"),      # FK to Product Dim
            col("user_id"),         # FK to User Dim (if you had one)
            col("brand"),           # Degenerate Dimension
            col("category_code"),   # Degenerate Dimension
            col("event_type"),      # Filter Dimension
            col("price").alias("transaction_amount") # <--- THE FACT (Raw Price)
        )
        # We do NOT aggregate here. We pass the granular data to the Dashboard.
    )

In [0]:
# =========================================================================
# 2. DIMENSION TABLE: PRODUCT HISTORY (SCD Type 2)
# =========================================================================
# RUBRIC: "Need to implement SCD-2"
# This tracks the "Official Catalog Price" changes over time.

@dlt.view(name="stg_products_cdc")
def stg_products_cdc():
    return (
        spark.readStream.table(SILVER_TABLE)
        .select("product_id", "price", "category_code", "brand", "event_time")
    )

dlt.create_streaming_table(
    name="gold_dim_products",
    comment="Product Dimension with Price History (SCD Type 2).",
    table_properties={"quality": "gold"}
)

dlt.apply_changes(
    target = "gold_dim_products",
    source = "stg_products_cdc",
    keys = ["product_id"],
    sequence_by = col("event_time"),
    stored_as_scd_type = 2,
    track_history_column_list = ["price", "category_code", "brand"]
)    